In [1]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "intfloat/e5-small-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
max_length = 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})

{'model_name': 'intfloat/e5-small-v2', 'dataset': 'glue/stsb', 'split': 'validation', 'device': 'mps', 'batch_size': 128, 'max_length': 128, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

{'num_examples': 1500, 'columns': ['sentence1', 'sentence2', 'label']}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   

                                  sentence2  label  
0      A man wearing a hard hat is dancing.   5.00  
1                A child is riding a horse.   4.75  
2  The man is feeding a mouse to the snake.   5.00  
3                  A man is playing guitar.   2.40  
4                 A man is playing a flute.   2.75  


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()
print(model_name)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

In [ ]:
def average_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    masked = last_hidden_state * mask
    summed = masked.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def encode_e5(texts, prefix, batch_size=64, max_length=128):
    prefixed = [f"{prefix}: {text}" for text in texts]
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(prefixed), batch_size):
            batch_texts = prefixed[i:i + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = model(**encoded)
            pooled = average_pool(outputs.last_hidden_state, encoded["attention_mask"])
            pooled = F.normalize(pooled, p=2, dim=1)
            all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = encode_e5(sentences1, prefix="query", batch_size=batch_size, max_length=max_length)
emb2 = encode_e5(sentences2, prefix="passage", batch_size=batch_size, max_length=max_length)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5_raw = 2.5 * (cosine_similarity + 1.0)
predicted_score_0_5_clipped = np.clip(predicted_score_0_5_raw, 0.0, 5.0)

norms1 = np.linalg.norm(emb1, axis=1)
norms2 = np.linalg.norm(emb2, axis=1)

In [ ]:
pearson_corr_raw = pearsonr(predicted_score_0_5_raw, labels).statistic
spearman_corr_raw = spearmanr(predicted_score_0_5_raw, labels).statistic
pearson_corr_clipped = pearsonr(predicted_score_0_5_clipped, labels).statistic
spearman_corr_clipped = spearmanr(predicted_score_0_5_clipped, labels).statistic

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5_raw"] = predicted_score_0_5_raw
results_df["predicted_score_0_5_clipped"] = predicted_score_0_5_clipped
results_df["embedding_norm_s1"] = norms1
results_df["embedding_norm_s2"] = norms2

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5_raw", "predicted_score_0_5_clipped", "embedding_norm_s1", "embedding_norm_s2"]].head(10))

In [ ]:
runtime_seconds = time.time() - start_time

norm_summary = pd.DataFrame({
    "query_embedding_norm": norms1,
    "passage_embedding_norm": norms2,
}).agg(["mean", "std", "min", "max"])

clip_low_count = int((predicted_score_0_5_raw < 0.0).sum())
clip_high_count = int((predicted_score_0_5_raw > 5.0).sum())
clip_any_count = int(((predicted_score_0_5_raw < 0.0) | (predicted_score_0_5_raw > 5.0)).sum())

quantiles = [0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
quantile_index = [f"q{int(q*100):02d}" for q in quantiles]
quantile_table = pd.DataFrame({
    "predicted_raw": pd.Series(predicted_score_0_5_raw).quantile(quantiles).to_numpy(),
    "predicted_clipped": pd.Series(predicted_score_0_5_clipped).quantile(quantiles).to_numpy(),
    "label": pd.Series(labels).quantile(quantiles).to_numpy(),
}, index=quantile_index)

range_bins = [0, 1, 2, 3, 4, 5]
range_labels = ["[0,1]", "(1,2]", "(2,3]", "(3,4]", "(4,5]"]
bucket_series = pd.cut(predicted_score_0_5_clipped, bins=range_bins, labels=range_labels, include_lowest=True)
range_summary = results_df.assign(pred_bucket=bucket_series).groupby("pred_bucket", observed=False).agg(
    count=("label", "size"),
    mean_pred=("predicted_score_0_5_clipped", "mean"),
    mean_label=("label", "mean"),
    min_pred=("predicted_score_0_5_clipped", "min"),
    max_pred=("predicted_score_0_5_clipped", "max"),
).reset_index()

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print("input_format_sentence1: query: <text>")
print("input_format_sentence2: passage: <text>")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pearson_correlation_raw: {pearson_corr_raw:.6f}")
print(f"spearman_correlation_raw: {spearman_corr_raw:.6f}")
print(f"pearson_correlation_clipped: {pearson_corr_clipped:.6f}")
print(f"spearman_correlation_clipped: {spearman_corr_clipped:.6f}")
print(f"clip_low_count: {clip_low_count}")
print(f"clip_high_count: {clip_high_count}")
print(f"clip_any_count: {clip_any_count}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
print("embedding_norm_summary:")
print(norm_summary)
print("predicted_score_quantiles:")
print(quantile_table)
print("calibration_range_summary:")
print(range_summary)